In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

    

import torchvision
import torchvision.transforms as transforms


import sys
sys.path.append('..')
from models import *   # bring everything in the folder models

global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')
device = torch.device("cuda" if use_gpu else "cpu") 
    
batch_size = 128

model_name = "cifar10"
#model = cifar10()
model = VGG16()

        

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.view(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)





=> Building model...
Files already downloaded and verified
Files already downloaded and verified


In [2]:
fdir = 'result/'+str(model_name)+'/model_best.pth.tar'
checkpoint = torch.load(fdir)
model.load_state_dict(checkpoint['state_dict'])


criterion = nn.CrossEntropyLoss().cuda()

model.eval()
model.cuda()

print("Initial model (HW3) accuracy")
prec = validate(testloader, model, criterion)

print("Initial model (HW3) first conv layer weights sum")
abs_sum = model.features[0].weight.abs().sum().item()
print(abs_sum)

Initial model (HW3) accuracy
Test: [0/79]	Time 0.429 (0.429)	Loss 0.2345 (0.2345)	Prec 94.531% (94.531%)
 * Prec 91.280% 
Initial model (HW3) first conv layer weights sum
302.16412353515625


In [ ]:
fdir = 'result/'+str(model_name)+'/model_best.pth.tar'
checkpoint = torch.load(fdir)
model.load_state_dict(checkpoint['state_dict'])

save_dir = 'result/'+str(model_name) + '_newloss_y1'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# number of epochs to train the model
n_epochs = 20  
# per epoch, all the training data set is used once

y=1 # step 2 with new loss term

lr = 0.01

optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)


model.train() # prep model for training


for epoch in range(n_epochs):
    # monitor training loss
    train_loss = 0.0
    train_loss1 = 0.0
    train_loss2 = 0.0
    
    ###################
    # train the model #
    ###################
    for data, target in trainloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        # clear the gradients of all optimized variables
        optimizer.zero_grad()
        # forward pass: compute predicted outputs by passing inputs to the model
        output = model(data)
        # calculate the loss
        
        loss1 = criterion(output, target)
        loss2 = model.features[0].weight.abs().sum() # + model.conv2.weight.abs().sum()

        #loss = loss1
        loss = loss1 + y*loss2
        
        loss.backward()
        optimizer.step()
        train_loss1 += loss1.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        train_loss2 += loss2.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        train_loss  += loss.item()*data.size(0) # as loss is tensor, .item() needed to get the value
        
    # print training statistics 
    # calculate average loss over an epoch
    train_loss1 = train_loss1/len(trainloader.dataset)
    train_loss2 = train_loss2/len(trainloader.dataset)
    train_loss = train_loss/len(trainloader.dataset)

    print('Epoch: {} \tTraining Loss1: {:.6f}'.format(epoch+1, train_loss1))
    print('Epoch: {} \tTraining Loss2: {:.6f}'.format(epoch+1, train_loss2))
    print('Epoch: {} \tTraining Loss: {:.6f}'.format(epoch+1, train_loss))
    scheduler.step()
    
# see following link for details of state_dict   
# https://pytorch.org/tutorials/recipes/recipes/saving_and_loading_a_general_checkpoint.html
torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': train_loss,
            }, save_dir+'/model_best.pth.tar')

In [21]:
checkpoint = torch.load(save_dir+'/model_best.pth.tar')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoch = checkpoint['epoch']
train_loss = checkpoint['loss']

model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 8219/10000 (82%)



In [22]:
abs_sum = model.features[0].weight.abs().sum().item()
print(abs_sum)

0.3348538875579834


In [ ]:
yList = [0.01, 0.1, 10, 100]

for y in yList:
    fdir = 'result/'+str(model_name)+'/model_best.pth.tar'
    checkpoint = torch.load(fdir)
    model.load_state_dict(checkpoint['state_dict'])
    
    save_dir = 'result/'+str(model_name) + '_newloss_y'+str(y)
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    # number of epochs to train the model
    n_epochs = 20  
    # per epoch, all the training data set is used once
        
    lr = 0.01
    
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    
    model.train() # prep model for training
    
    
    for epoch in range(n_epochs):
        # monitor training loss
        train_loss = 0.0
        train_loss1 = 0.0
        train_loss2 = 0.0
        
        ###################
        # train the model #
        ###################
        for data, target in trainloader:
            data, target = data.to(device), target.to(device) # loading to GPU
            # clear the gradients of all optimized variables
            optimizer.zero_grad()
            # forward pass: compute predicted outputs by passing inputs to the model
            output = model(data)
            # calculate the loss
            
            loss1 = criterion(output, target)
            loss2 = model.features[0].weight.abs().sum() # + model.conv2.weight.abs().sum()
    
            #loss = loss1
            loss = loss1 + y*loss2
            
            loss.backward()
            optimizer.step()
            train_loss1 += loss1.item()*data.size(0) # as loss is tensor, .item() needed to get the value
            train_loss2 += loss2.item()*data.size(0) # as loss is tensor, .item() needed to get the value
            train_loss  += loss.item()*data.size(0) # as loss is tensor, .item() needed to get the value
            
        # print training statistics 
        # calculate average loss over an epoch
        train_loss1 = train_loss1/len(trainloader.dataset)
        train_loss2 = train_loss2/len(trainloader.dataset)
        train_loss = train_loss/len(trainloader.dataset)
    
        print('Epoch: {} \tTraining Loss1: {:.6f}'.format(epoch+1, train_loss1))
        print('Epoch: {} \tTraining Loss2: {:.6f}'.format(epoch+1, train_loss2))
        print('Epoch: {} \tTraining Loss: {:.6f}'.format(epoch+1, train_loss))
        scheduler.step()
    # see following link for details of state_dict   
    # https://pytorch.org/tutorials/recipes/recipes/saving_and_loading_a_general_checkpoint.html
    torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': train_loss,
                }, save_dir+'/model_best.pth.tar')

In [27]:
save_dir = 'result/'+str(model_name) + '_newloss_y'

for y in yList:
    checkpoint = torch.load(save_dir+str(y)+'/model_best.pth.tar')
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    epoch = checkpoint['epoch']
    train_loss = checkpoint['loss']
    
    model.eval()
    
    test_loss = 0
    correct = 0
    
    with torch.no_grad():
        for data, target in testloader:
            data, target = data.to(device), target.to(device) # loading to GPU
            output = model(data)
            pred = output.argmax(dim=1, keepdim=True)  
            correct += pred.eq(target.view_as(pred)).sum().item()
    
    test_loss /= len(testloader.dataset)

    print("Loss with y=",y)
    print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
            correct, len(testloader.dataset),
            100. * correct / len(testloader.dataset)))
    abs_sum = model.features[0].weight.abs().sum().item()
    print(abs_sum)

Loss with y= 0.01

Test set: Accuracy: 9103/10000 (91%)

1.841234564781189
Loss with y= 0.1

Test set: Accuracy: 8983/10000 (90%)

0.229983389377594
Loss with y= 10

Test set: Accuracy: 7929/10000 (79%)

1.3596811294555664
Loss with y= 100

Test set: Accuracy: 7475/10000 (75%)

11.79818058013916


| Case | Cost function | Test accuracy (%) | Conv1 weight abs sum |
|------|---------------|-------------------|----------------------|
| 1. Initial version | loss1 | 91.280% |302.1641 |
| 2. New loss term (γ = 1) | loss1 + loss2 | 82% | 0.3349 |
| 3. Fine tuned (optimal y = 0.1) | loss1 + y*loss2 (y = 0.1) | 90%  | 0.229983389377594 |

The test accuraccy tends to decrease when increasing the regularization factor. The weight sum tends to decrease when increasing the regularization factor. However, at large values of y, the weight sum starts to go up again. This could be since the large y makes it so the weights are prioritized over the accuracy. There may also be some randomness involved in the training that may have effects on the weight sum and accuracy. The large y may also affect the optimizer step to cause overshoot.